### Student

Name: Vladimir Ungureanu <br>
Badge Number: 2141678 <br>
Master Program: Physics of Data <br>

## Domain: Applications of Machine Learning in Biomedicine

This project operates in the domain of **machine learning applied to biomedical
research**, using open-access articles from **PubMed Central (PMC)** published
within the last 12 months.

### Scope

The corpus covers the intersection of ML methods and biomedical problems,
including:

- **Clinical prediction** — diagnosis support, prognosis, disease risk scoring
- **Medical imaging** — segmentation, detection, and classification in
  radiology, pathology, and dermatology
- **Drug discovery & genomics** — molecule property prediction, variant
  calling, protein structure and function
- **Clinical NLP** — information extraction from EHRs, discharge summaries,
  and biomedical literature
- **ML methodology in biomedicine** — interpretability, federated learning,
  handling class imbalance and missing clinical data

### Corpus

| Property | Value |
|---|---|
| Source | PubMed Central (open access subset) |
| Query | `"machine learning"[Title/Abstract] AND open access[filter]` |
| Time window | Last 12 months |
| Selection | 500 full-text articles — the **most relevant** results from the top-1000 relevance-sorted PMC hits |
| Format | BioC XML → plain text → chunked |

### Why separate BioC XML markup from text and metadata?

The BioC API returns articles wrapped in XML markup (`<passage>`, `<text>`,
`<infon>`, `<offset>`...). The pipeline stores clean passage prose separately
from its structural metadata:

- **Embeddings do not receive XML tags.** The sentence-transformer model sees
  natural language prefixed only by the article title, semantic section label,
  and passage type.
- **Chunking uses clean input.** Token limits are measured over prose rather
  than XML syntax, while tokenizer character spans are retained for every
  chunk.
- **Structure is preserved.** Each article has a metadata sidecar containing
  its title, article identifiers, section types, passage types, original BioC
  offsets, and positions in the flattened text.
- **Citations remain traceable.** Chunk metadata is propagated through FAISS,
  retrieval, generation, and evaluation.

This design keeps the storage and modeling benefits of plain text without
losing article-level provenance.

Unzip the code package (`biomed_rag.zip`) and any data archives uploaded to Colab. Data archives are only extracted if their folder does not exist yet.

In [1]:
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    content = Path("/content")
    for name in ["biomed_rag", "pmc_xml_test", "pmc_raw_texts", "document_repository", "vector_database"]:
        archive = content / f"{name}.zip"
        if not archive.exists() or (name != "biomed_rag" and (content / name).exists()):
            continue
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(content)
        print(f"Unzipped {archive.name}")

Unzipped biomed_rag.zip
Unzipped pmc_xml_test.zip


Install dependences not already installed in Google Colab. `bitsandbytes` is only needed when a model is loaded in 4-bit (`load_in_4bit=True`, off by default).

In [1]:
%pip install -q faiss-cpu rank-bm25 accelerate "bitsandbytes>=0.46.1"

## Libraries

| Library | Purpose |
|---|---|
| `sentence-transformers` | Sentence embedding — encodes text chunks into dense vectors; cross-encoder reranking of retrieved candidates |
| `faiss` | Vector store — fast similarity search over chunk embeddings |
| `rank-bm25` | Lexical retrieval — BM25 scores fused with the dense results |
| `transformers` (`AutoModelForCausalLM`, `AutoTokenizer`) | Loads the pretrained LLM used for generation |
| `torch` | Deep learning backend — tensor ops and model inference (incl. quantization support) |


## Project structure

The pipeline code lives in the `biomed_rag/` package. This notebook runs it step by step and presents the results.

| Module | Responsibility |
|---|---|
| `config.py` | Paths, model IDs and PMC query settings |
| `utils.py` | JSON files and archives |
| `corpus.py` | PMC search, BioC XML download, section filtering, text extraction |
| `chunking.py` | Paragraph-aware chunking into ≤512-token segments |
| `stats.py` | Corpus and chunking statistics |
| `indexing.py` | Chunk embeddings and the FAISS index |
| `retrieval.py` | Hybrid retrieval: dense + BM25, fusion, reranking, merging |
| `llm.py` | Loading the language models, greedy generation, memory cleanup |
| `prompts.py` | All prompt templates |
| `testset.py` | Question and reference-answer generation |
| `rag.py` | RAG prompt and answering |
| `evaluation.py` | Retrieval metrics, LLM judge and the Markdown report |



In [1]:
from biomed_rag import chunking, config, corpus, evaluation, indexing, prompts, rag, retrieval, stats, testset, utils
from biomed_rag.llm import LLM, free_memory
from biomed_rag.utils import make_zip, save_json

### IDs search

The search returns the top 1,000 most relevant PMC IDs for "machine learning"
published in the last year.

In [6]:
query = corpus.build_pmc_query(days=365)
pmc_ids = corpus.search_pmc_ids(query, max_results=1000)
save_json(pmc_ids, config.PMC_IDS_PATH)
print("Found PMC IDs:", len(pmc_ids))
print(pmc_ids[:10])

Found PMC IDs: 1000
['13337728', '13485820', '13342631', '13232287', '12848914', '12607989', '12538836', '12590498', '12699858', '13185850']


### XLM download

Download the first 500 availabale XLMs of the found IDs

In [7]:
failed_xml = corpus.download_corpus(pmc_ids, config.XML_DIR, n_articles=500)

### Find XLM unique sections



In [5]:
if config.XML_DIR.exists():
    corpus.print_section_counts(corpus.count_section_types(config.XML_DIR))

## Excluding boilerplate sections

The BioC API labels every passage with `infon["section_type"]` (the section) and
`infon["type"]` (the passage role). After inspecting the labels used across a
200 article samples, they are filtered in 2 ways:

**By `section_type` — whole sections dropped:**

| `section_type` | What it contains | Passages (200-article sample) |
|---|---|---|
| `REF` | Bibliography / references | 12,622 |
| `ABBR` | Abbreviation lists | 1,359 |
| `SUPPL` | Supplementary material | 952 |
| `REVIEW_INFO` | Editorial metadata (received/accepted dates, editors) | 582 |
| `AUTH_CONT` | Author contributions | 532 |
| `COMP_INT` | Conflict of interest statements | 509 |
| `ACK_FUND` | Funding / acknowledgements | 149 |
| `APPENDIX` | Appendix material | 88 |

**By `type` - only the raw markup is dropped:**

| `type` | What it contains |
|---|---|
| `table` | Embedded `<table>` XML markup and algorithm pseudocode |

Table **captions** (`table_caption`) and **footnotes** (`table_footnote`) are
kept as they are short, readable prose that can help retrieval, while the raw
`<table>` XML itself is removed because even though it contains structured data,
it is not easily readable by LLMs. Figure captions (`FIG`) are kept for the same
reason as the table footnotes.

**Kept sections:** `TITLE`, `ABSTRACT`, `INTRO`, `METHODS`, `RESULTS`,
`DISCUSS`, `CONCL`, `FIG`(figure descriptions are informative), `TABLE` (captions/footnotes only).

In [3]:
corpus.filter_sections(
    config.XML_DIR,
    config.FILTERED_XML_DIR,
    excluded_sections=config.EXCLUDED_SECTIONS,
    excluded_types=config.EXCLUDED_TYPES,
)

Wrote 500 filtered XML files to /content/pmc_xml_filtered/
Passages kept: 44390, removed: 38262


(44390, 38262)

In [4]:
corpus.extract_corpus_text(config.FILTERED_XML_DIR, config.RAW_TEXT_DIR)


Done. 500 articles in /content/pmc_raw_texts/


500

## Paragraph-aware chunking into 512-token segments

Each article is chunked on its own, so a chunk never mixes two articles:

1. **Tokenization**: the article is tokenized with the embedding model's tokenizer, keeping the character offsets of every token.
2. **Paragraph-aware windows**: a chunk holds at most **512 tokens** and ends at the last paragraph boundary inside that window. Boundaries that would leave a chunk shorter than 128 tokens are ignored; if no boundary qualifies, the window is cut at a word boundary.
3. **Overlap**: consecutive chunks overlap by up to **64 tokens**, starting on a word boundary.
4. **Short tails**: if the last chunk of an article would be shorter than 128 tokens, it is replaced by the final 512 tokens of the article.
5. **Original text**: every chunk is a slice of the original text, taken through the token offsets rather than decoded from token IDs, so casing, numbers and symbols are preserved.

Chunks are saved as `<PMC_ID>_chunk_<NNNN>.txt`, numbered per article.

The same tokenizer as the embedding model used later - [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings). This guarantees that "512 tokens" is measured in the exact units the embedding model will consume (its `max_seq_length` is 512). Because that limit also counts the `[CLS]` and `[SEP]` tokens, a full 512-token chunk loses its last two tokens when it is embedded, that is why 510 was used.

In [5]:
chunking.chunk_corpus(
    config.RAW_TEXT_DIR,
    config.CHUNK_DIR,
    tokenizer_id=config.EMBEDDING_MODEL_ID,
    chunk_tokens=510,
    overlap_tokens=64,
    min_chunk_tokens=128,
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Generated 10588 chunks in /content/document_repository.


10588

### Corpus and Chunking Statistics

The final corpus contains **500 biomedical articles**, all of which are represented in the chunk collection. The articles are divided into **10,588 text chunks**, corresponding to an average of approximately **21.18 stored chunks per document**.

“Content tokens” count only the chunk text, whereas “tokens with special tokens” include the `[CLS]` and `[SEP]` tokens added by the embedding model.

#### Corpus overview

| Metric | Value |
|---|---:|
| Raw documents | 500 |
| Documents represented in chunks | 500 |
| Total chunks | 10,588 |
| Average chunks per document | 21.18 |

#### Chunk-level statistics

| Metric | Mean | Median | Standard deviation | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Characters per chunk | 2,278.1 | 2,340 | 498.9 | 415 | 3,848 |
| Words per chunk | 314.4 | 326 | 66.9 | 47 | 459 |
| Content tokens per chunk | 420.6 | 441 | 79.8 | 131 | 511 |
| Tokens with special tokens | 422.6 | 443 | 79.8 | 133 | 513 |

#### Document-level statistics

| Metric | Mean | Median | Standard deviation | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Chunks per document | 21.2 | 20.0 | 9.4 | 1 | 75 |
| Characters per raw document | 40,977.6 | 37,624 | 18,450.7 | 1,490 | 144,465 |
| Words per raw document | 5,661.0 | 5,210.5 | 2,512.9 | 206 | 19,395 |
| Tokens per raw document | 7,583.8 | 6,939 | 3,307.0 | 239 | 25,291 |

#### Validation

| Check | Result |
|---|---:|
| Mixed-document chunks | 0 |
| Chunks over 512 content tokens | 0 |
| Inputs over the model limit of 512 tokens | 1 |
| Documents without chunks | 0 |
| Chunks with unknown document IDs | 0 |

Although no chunk exceeds the configured content limit, one embedding input exceeds the model’s 512-token limit:

```text
PMC12894987_chunk_0013.txt: 513 tokens

In [8]:
corpus_stats = stats.corpus_statistics(config.CHUNK_DIR, config.RAW_TEXT_DIR, tokenizer_id=config.EMBEDDING_MODEL_ID)
stats.print_corpus_report(corpus_stats)
save_json(corpus_stats, config.PROFILE_PATH)
print(f"\nSaved statistics to {config.PROFILE_PATH.name}")

=== CORPUS OVERVIEW ===
Raw documents:                   500
Documents represented in chunks:  500
Total chunks:                    10588
Average chunks per document:     21.18

=== CHUNK-LEVEL STATISTICS ===
Characters per chunk                 n=10588  mean=   2278.1  median=  2340.0  std=   498.9  min=   415  max=  3848
Words per chunk                      n=10588  mean=    314.4  median=   326.0  std=    66.9  min=    47  max=   459
Content tokens per chunk             n=10588  mean=    420.6  median=   441.0  std=    79.8  min=   131  max=   511
Tokens with special tokens           n=10588  mean=    422.6  median=   443.0  std=    79.8  min=   133  max=   513

=== DOCUMENT-LEVEL STATISTICS ===
Chunks per document                  n=  500  mean=     21.2  median=    20.0  std=     9.4  min=     1  max=    75
Characters per raw document          n=  500  mean=  40977.6  median= 37624.0  std= 18450.7  min=  1490  max=144465
Words per raw document               n=  500  mean=   5661.0

### Choice of Embedding Model: `NeuML/pubmedbert-base-embeddings`

This project uses [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings) to represent document chunks and user queries as dense vectors. The model was selected because it is specifically designed for semantic similarity and retrieval in biomedical literature, which closely matches this project's corpus of PubMed Central articles about machine learning in biomedicine.

The model is based on **PubMedBERT**, whose pretraining data consists of biomedical abstracts and full-text articles. It was subsequently fine-tuned with the `sentence-transformers` framework using PubMed title–abstract pairs and related title pairs. Consequently, its vector representations are better adapted to biomedical terminology, abbreviations, and relationships than those produced by general-purpose embedding models.

The main reasons for selecting this model are:

- **Domain relevance:** its biomedical training data is closely aligned with the project's PMC corpus.
- **Semantic retrieval:** it is explicitly fine-tuned to map semantically related sentences and paragraphs to nearby points in the embedding space.
- **Biomedical vocabulary:** it can represent specialized clinical and scientific terminology more effectively than a general-purpose model.
- **Suitable output size:** each text is mapped to a **768-dimensional vector**, providing detailed semantic representations while remaining practical for FAISS indexing.
- **Sequence length:** its maximum sequence length of **512 tokens** is appropriate for the document chunks used by the retrieval pipeline.
- **Retrieval performance:** according to the model card, it outperforms several general-purpose and biomedical alternatives on the reported PubMed semantic-similarity evaluations.

In [7]:
indexing.build_index(config.CHUNK_DIR, config.VECTOR_DB_DIR, model_id=config.EMBEDDING_MODEL_ID, batch_size=128)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 10588 chunks with title and section metadata...


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

FAISS index built: 10588 vectors, dim 768


<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x7b5697891bb0> >

### Hybrid retrieval

Retrieval combines two rankings of the chunks:

- **Dense:** cosine similarity between PubMedBERT embeddings, searched with FAISS.
- **Lexical:** BM25 scores, which catch exact terms such as metric names, gene names and acronyms that embeddings can miss.

The two rankings are merged with reciprocal rank fusion (RRF), and a cross-encoder (`BAAI/bge-reranker-base`) reranks the top candidates. Duplicate chunks are dropped, a chunk adjacent to an already selected chunk of the same article is merged into it, and each hit is extended with its neighbouring chunks. If the best dense similarity is below `min_dense`, no passages are returned and the generator is told that no documents were retrieved.

### Choice of Reranking Model: `BAAI/bge-reranker-base`

Dense retrieval and BM25 first identify a relatively small set of potentially relevant chunks. The reranker then assigns a more precise relevance score to each question–passage pair and reorders the candidates before the final passages are given to the generative model.

Unlike the embedding model, which encodes the question and each passage independently, `bge-reranker-base` is a **cross-encoder**. It processes the question and passage together, allowing attention between their tokens. This makes it better able to recognize whether a passage directly answers the question, rather than merely discussing a semantically related topic. The additional precision is useful in biomedical retrieval, where several chunks may contain similar terminology while only one contains the requested metric, method, gene, result, or comparison.

The main reasons for selecting this model are:

- **Purpose-built reranking:** it is specifically designed to rerank a small candidate set returned by a faster first-stage retriever.
- **More precise relevance estimation:** joint question–passage processing can capture exact relationships that independent embeddings may miss.
- **Suitable for hybrid retrieval:** it can refine candidates found through both semantic similarity and exact-term BM25 matching.
- **Good efficiency–quality balance:** the model has approximately **278 million parameters** and is based on **XLM-RoBERTa-Base**, making it substantially lighter than `bge-reranker-large` while still providing cross-encoder reranking.
- **Practical for Google Colab:** only the top fused candidates are reranked, rather than the entire corpus, which keeps the additional computational cost manageable.
- **No domain-specific retraining required:** although it is not trained specifically for biomedicine, it can compare English questions and passages directly, while the biomedical PubMedBERT model remains responsible for domain-specific first-stage retrieval.


### Retrieval configuration

| Parameter | Value | Motivation |
| --- | --- | --- |
| k_final | 5 | Returns at most five passage groups to the generator. This provides several sources of evidence without adding excessive context. |
| k_dense | 30 | Retrieves 30 semantic candidates with PubMedBERT and FAISS. A relatively broad candidate set improves recall when the correct chunk is relevant but not ranked first. |
| k_bm25 | 30 | Retrieves 30 lexical candidates. BM25 complements dense retrieval by matching exact biomedical terminology, acronyms, gene names, model names, and metrics such as RMSE or CV. |
| k_rerank | 30 | Sends only the 30 best fused candidates to the cross-encoder. This preserves a broad candidate pool while limiting the more expensive reranking step. |
| rrf_k | 60 | Controls reciprocal rank fusion. The conventional value of 60 reduces sensitivity to small rank differences and rewards chunks retrieved by both dense search and BM25. |
| min_dense | 0.4 | Refuses retrieval when the best dense cosine similarity is below 0.45. This reduces the risk of answering from unrelated passages.|
| expand_neighbours | True | Adds the chunks immediately before and after a selected chunk when they belong to the same article. This recovers evidence split across chunk boundaries. |
| use_reranker | True | Enables cross-encoder reranking, which compares each question and passage jointly and can distinguish directly answering passages from merely related ones. |

In [6]:
retrieval_config = retrieval.RetrievalConfig(
    k_final=2,#bigger k_final would result in better retrieval but exceeds GPU memory
    k_dense=30,
    k_bm25=30,
    k_rerank=30,
    rrf_k=60,
    min_dense=0.4,#started with 0.45 but noticed that some answerable questions were not answered
    expand_neighbours=True,
    use_reranker=True,
)
retriever = retrieval.HybridRetriever.from_disk(config.VECTOR_DB_DIR, config.CHUNK_DIR, retrieval_config)

for hit in retriever.search("What techniques are used to determine the particle size distribution and structural characteristics of catalysts?"):
    print(f"{hit['score']:.3f}  {hit['chunk_file']}")
    print(hit["text"][:200], "\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building BM25 index over metadata-enriched chunk texts...
BM25 index ready: 10588 chunks


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

0.973  PMC12660924_chunk_0018.txt
lattice spacing (i.e., crystal expansion) that results in a wide band gap which hinders electron transitions between energy states and results in poorer electrochemical performance. Therefore, XRD mod 

0.961  PMC12660924_chunk_0036.txt
mappings, enabling the generation of missing spectroscopic modalities from the available ones, thereby mitigating challenges associated with incomplete multimodal datasets. In its current form, our st 



### Choice of Generative Model: `LiquidAI/LFM2-1.2B-RAG`

This project uses [`LiquidAI/LFM2-1.2B-RAG`](https://huggingface.co/LiquidAI/LFM2-1.2B-RAG) as the generative component of the retrieval-augmented generation pipeline. The model receives the document chunks retrieved from the FAISS index and generates an answer grounded in that context.

The model was selected primarily because it is specifically fine-tuned for **retrieval-augmented generation**. Unlike a general-purpose language model, it is designed to answer questions from supplied documents.

The main reasons for selecting this model are:

- **RAG specialization:** the model is trained to identify and use relevant information from documents included in its prompt.
- **Grounded question answering:** conditioning generation on retrieved evidence helps reduce unsupported answers compared with relying only on the model's internal knowledge.
- **Compact model size:** with approximately **1.2 billion parameters**, it is substantially smaller than many general-purpose LLMs and is therefore suitable for environments with limited computational resources, such as Google Colab.

In this pipeline, the embedding model first converts the user's question into a vector. FAISS retrieves the most semantically relevant biomedical text chunks, and these chunks are inserted into the prompt expected by `LFM2-1.2B-RAG`. The model then produces an answer using the retrieved passages as evidence. This division of responsibilities combines the domain-specific retrieval capabilities of `NeuML/pubmedbert-base-embeddings` with a lightweight model specialized for context-grounded generation.

In [10]:
print(prompts.RAG_SYSTEM_TEMPLATE)

with rag.RAGPipeline.load(retriever, model_id=config.GENERATOR_MODEL_ID) as generator:
    result = generator.answer("What techniques are used to determine the particle size distribution and structural characteristics of catalysts?", k=retrieval_config.k_final, max_new_tokens=256)
print("\n" + result["answer"])

The following documents may provide information relevant to the user's query:
{documents}

Answer using only the supplied documents. Cite each supported claim with its source label, such as [doc1]. Do not invent citations.
Loading LiquidAI/LFM2-1.2B-RAG...


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/91.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded successfully!


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.



The techniques used to determine the particle size distribution and structural characteristics of catalysts include:  
- **SAXS (Small-Angle X-ray Scattering)**: Measures scattering at small angles (0.1–5°) to analyze particle size and distribution.  
- **XRD (X-ray Diffraction)**: Reveals crystalline structure, phase, lattice constants, strain, site occupancy, and atomic displacement.  
- **PDF (Pair Distribution Function)**: Analyzes short-range atomic arrangements (1–10 nm) in catalysts.  
- **XAS (X-ray Absorption Fine Structure)**: Provides atomic-level information on oxidation states, bonding, and coordination.  
- **HAXPES (High Energy X-ray Photoelectron Spectroscopy)**: Determines elemental composition, chemical states, and surface electronic environments.  
- **XAFS (X-ray Absorption Fine Structure)**: Investigates valence band structure and its correlation with catalytic performance.  
These techniques collectively characterize catalyst size, shape, crystallinity, and elect

In [11]:
free_memory(report=True)
!nvidia-smi

CUDA Memory Allocated: 1490.06 MB
CUDA Memory Reserved: 1726.00 MB
Sat Oct 17 13:27:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             40W /   70W |    1863MiB /  15360MiB |      0%      Default |
|                                         |                        |     

### Choice of Q/A Generation Model: `Qwen/Qwen3-4B-Instruct-2507`

This project uses [`Qwen/Qwen3-4B-Instruct-2507`](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) to generate questions and answers from the collected biomedical documents. The model receives a passage from a PubMed Central article and is instructed to produce Q/A pairs whose answers can be supported by the source text.

The model was selected because it provides a strong balance between **generation quality, instruction-following ability, context capacity, and computational cost**. With approximately **4 billion parameters**, it is more capable than very small language models while remaining practical to run in a constrained environment.

The main reasons for choosing this model are:

- **Strong instruction following:** the model is post-trained to follow natural-language instructions, which is important for controlling the required Q/A format.
- **Good text comprehension:** generating valid Q/A pairs requires identifying the main concepts, methods, results, and relationships described in a passage.
- **Improved scientific reasoning:** the model card reports improvements in science, logical reasoning, and knowledge-oriented tasks, which are relevant when processing biomedical literature.

#### Initial model: `Mistral-7B-Instruct-v0.2`

The first model used for Q/A generation was [`mistralai/Mistral-7B-Instruct-v0.2`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2). It was chosen because of its strong reputation as a 7B-parameter instruction-tuned model and its ability to follow moderately complex prompts. However, in this project it produced unsatisfactory question/answer pairs for evaluation purposes.

Despite extensive prompt engineering — including explicit instructions to write self-contained questions, avoid referring to the source text, and generate answers that do not depend on the article context — the model consistently produced outputs that contained phrases such as:

- "the article"
- "the study"
- "the authors"
- "the paper"

These references made the generated questions context-dependent rather than self-contained. Because the goal of the evaluation is to simulate a user asking a question *without* access to the original article, such phrasing leaks information about the source and biases retrieval and answering. A self-contained question should be answerable from the retrieved context alone, not from knowledge of the article.

This issue persisted across multiple prompt variations.

For this reason, generation was switched to `Qwen/Qwen3-4B-Instruct-2507`. In direct comparison, Qwen3-4B produced more self-contained questions with fewer source-referencing artifacts, while still maintaining good biomedical comprehension and instruction following. It also offered a smaller memory footprint than Mistral-7B, which was beneficial given the available compute resources.

### Prompt Engineering for Question Generation

The question-generation prompt was designed to create realistic evaluation queries from passages sampled from the biomedical document repository. The objective is not simply to reformulate a sentence as a question, but to simulate how a student would query the RAG system without having seen the source article.

The prompt is divided into four clearly labelled sections: **background**, **task**, **constraints**, and **output format**. This structure separates the model's role from the generation requirements and makes the instructions easier to follow.

#### Background and role definition

The model is instructed to behave as a student seeking information from a RAG system. Importantly, the simulated student is described as having no knowledge of the underlying article, dataset, analysis, or database.

This framing prevents questions such as:

> What method was used in this study?

Although such a question may be answerable from the passage, it is not a realistic standalone query because the user has not identified which study is being discussed. Instead, the prompt encourages self-contained questions such as:

> What methods are commonly used to select features in high-dimensional biomedical data?

The role definition therefore makes the generated questions better representations of queries that a real user could submit to the system.

#### Explicit generation task

The prompt asks for exactly **one clear and direct question** based on the supplied PubMed passage. Generating one question at a time reduces formatting errors and avoids producing several closely related questions from the same chunk.

The instruction to frame the question as if the model had never seen the source text is particularly important. Without it, language models frequently produce questions containing artificial references such as *“according to the passage”* or *“in this study.”* These references reveal that the question was generated from a document and would make the evaluation less representative of real RAG usage.

#### Constraints on question content

Several negative constraints are included to control common failure modes:

- The question must concern a meaningful **tool, metric, gene, method, or biomedical concept** found in the passage.
- It must be phrased as a broad, standalone student query.
- It must not refer to a *study*, *paper*, *author*, *dataset*, *analysis*, or *passage*.
- It must not contain expressions such as *“according to the text”* or *“in this paper.”*
- Its answer must be present in the supplied passage.
- The model must not provide an answer, explanation, or commentary.

These constraints serve two purposes. First, they make each question understandable without access to its source. Second, they ensure that the source passage can be treated as the reference evidence when evaluating retrieval and answer generation.

Requiring answerability from the passage is essential. A question may be topically related to a chunk while still requiring external knowledge. Such a question would not provide a reliable test of whether the RAG system retrieved and used the correct evidence.

#### Structured output

The model is required to return the question inside XML-style tags and it was used to to make the generated questions consistent, automatically extractable, and easy to validate.

```text
<question>Your generated question</question>

In [2]:
print(prompts.QUESTION_GENERATION_TEMPLATE)

qa_llm = LLM.load(config.QA_MODEL_ID)
questions = testset.generate_questions(qa_llm, config.CHUNK_DIR, n_passages=20, seed=42)

BACKGROUND
You are a student trying to get answers from a RAG system by asking it broad questions. You do NOT have any context about the underlying study, the paper, the dataset, the specific analysis, or the database from which your information comes.

YOUR TASK
I will provide you with a passage of a Pubmed article. Your task is to generate 1 clear, direct question based on this text and frame it as you have never seen the text.

CONSTRAINTS
- Frame the question as a general student query. Ask about the tool, metric, gene, method, or concept broadly.
- You do NOT know that there is a 'study', 'dataset', 'paper', 'analysis', or 'author'. Never mention these words or refer to them.
- DO NOT refer to the text or creators of the text (do not use words like 'the passage', 'this study', 'the author', 'the paper', 'here', 'the dataset', 'the analysis').
- Do not output any answer, commentary, or explanation.
- Make sure the answear is contained in the passage.

OUTPUT FORMAT
Output the quest

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded successfully!
Sampled 20 passages.
[1/20] Generated Q: How can machine learning algorithms be used to improve the diagnosis of indeterm...
[2/20] Generated Q: What techniques are used to determine the particle size distribution and structu...
[3/20] Generated Q: What is the importance of capturing detailed and diverse clinical data in unders...
[4/20] Generated Q: What machine learning algorithms are commonly used to predict venous thromboembo...
[5/20] Generated Q: What is a commonly used tool for managing citations and facilitating the screeni...
[6/20] Generated Q: What role does rheological analysis play in controlling the texture and stabilit...
[7/20] Generated Q: What is the difference between logistic model trees and alternating decision tre...
[8/20] Generated Q: What metrics are used to evaluate both the accuracy and uncertainty of a traffic...
[9/20] Generated Q: What is the role of the enzyme homogentisic 1,2-dioxygenase in the metabolism of...
[10/20] Generate

### Prompt Engineering for Reference-Answer Generation

The answer-generation prompt is designed to produce a concise **reference answer** for each synthetic question. The answer must be grounded exclusively in the corresponding PubMed passage so that it can later serve as a reference when evaluating the RAG system.

The prompt is divided into four sections: **background**, **task**, **constraints**, and **output format**. This organization makes the role, source of evidence, expected behaviour, and output structure explicit.

#### Role definition

The prompt assigns the model the role of a precise biomedical question-answering assistant. This encourages the model to focus on factual information and use terminology appropriate for the biomedical domain.

The background also states that the assistant has no contextual knowledge of the article, study, dataset, or analysis from which the passage was taken. This discourages references to unspecified external material and helps produce an answer that can stand on its own.

#### Passage-grounded answering

The central instruction is:

> Answer the question using ONLY the provided passage.

This is the most important grounding constraint. Although the language model may possess relevant biomedical knowledge in its parameters, the reference answer must contain only information supported by the selected passage. Otherwise, the generated answer could include correct but externally derived facts that the RAG system cannot recover from the indexed corpus.

Using the passage as the sole source of evidence creates a controlled relationship between:

1. the generated question;
2. the source chunk;
3. the reference answer.

This relationship is required for meaningful retrieval and answer-quality evaluation.

#### Constraints

The constraints address common unwanted generation behaviours:

- **Use only directly stated facts:** the model should avoid unsupported inferences and information recalled from its internal knowledge.
- **Avoid source-related preambles:** expressions such as *“Based on the passage”* and *“According to the text”* are prohibited because the a user would not know what chunk the system is referring to.
- **Do not add commentary:** the model should return the answer itself rather than explanations about how it interpreted the question or followed the instructions.

Together, these restrictions aim to produce answers that are factual, direct, and suitable for comparison with the output of the RAG system.

#### Concise answer requirement

The output is explicitly described as a **concise answer**. Concision is useful because the generated output is intended to serve as a reference rather than a detailed article summary. A short answer:

- focuses on the information needed to answer the question;
- reduces the chance of introducing unsupported claims;
- facilitates comparison with the RAG-generated answer;
- avoids irrelevant background information.

The model is not necessarily required to copy a sentence verbatim. It may summarize or paraphrase the relevant information, provided that every fact remains supported by the passage.

#### Structured output

The answer must be enclosed in XML-style tags:

```text
<answer>Your concise answer</answer>

In [3]:
print(prompts.GOLD_ANSWER_TEMPLATE)
 
test_set = testset.generate_gold_answers(qa_llm, questions)
qa_llm.close()
save_json(test_set, config.TEST_SET_PATH)
 
print(f"Test set: {len(test_set)} questions in {config.TEST_SET_PATH.name}")

BACKGROUND
You are a precise biomedical question-answering assistant that answears from it's own knowledge. You do NOT have any context about the underlying study, the paper, the dataset, the specific analysis, or the database from which your information comes.

YOUR TASK
Answer the question using ONLY the provided passage.

CONSTRAINTS
- Answer using ONLY facts directly mentioned in the passage.
- Do not say 'Based on the passage' or 'According to the text'.
- Do not output any commentary.

OUTPUT FORMAT
Output the concise answer wrapped inside tags: <answer>Your concise answer</answer>

PASSAGE:
{passage}

QUESTION:
{question}

[1/20] Q: How can machine learning algorithms be used to imp... -> A: Machine learning algorithms can be used to improve the diagnosis of indeterminate thyroid nodules by integrating miRNA data from fine needle aspiration and analyzing it to enhance diagnostic precision.
[2/20] Q: What techniques are used to determine the particle... -> A: SAXS and PDF are use

In [4]:
test_set = testset.load_test_set(
    config.TEST_SET_PATH,
    config.CHUNK_DIR,
)

## RAG answering on the test set

Each test question goes through the full pipeline: hybrid retrieval followed by `LFM2-1.2B-RAG`. The answers and the retrieved passages are saved to `lfm_rag_answers.json`.

In [7]:
with rag.RAGPipeline.load(
    retriever,
    model_id=config.GENERATOR_MODEL_ID,
) as generator:
    answers = rag.answer_test_set(
        generator,
        test_set,
        k=2,
        max_new_tokens=100,
    )
 
save_json(answers, config.RAG_ANSWERS_PATH)


Loading LiquidAI/LFM2-1.2B-RAG...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!
Generating RAG answers for 20 questions...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[1/20] Q: How can machine learning algorithms be used to imp... -> Answered.
[2/20] Q: What techniques are used to determine the particle... -> Answered.
[3/20] Q: What is the importance of capturing detailed and d... -> Answered.
[4/20] Q: What machine learning algorithms are commonly used... -> Answered.
[5/20] Q: What is a commonly used tool for managing citation... -> Answered.
[6/20] Q: What role does rheological analysis play in contro... -> Answered.
[7/20] Q: What is the difference between logistic model tree... -> Answered.
[8/20] Q: What metrics are used to evaluate both the accurac... -> Answered.
[9/20] Q: What is the role of the enzyme homogentisic 1,2-di... -> Answered.
[10/20] Q: How can machine learning algorithms be used to gen... -> Answered.
[11/20] Q: What are the key factors considered when assessing... -> Answered.
[12/20] Q: Which machine learning models are known for mainta... -> Answered.
[13/20] Q: What image resolution is recommended for machine l... -> Answe

PosixPath('/content/lfm_rag_answers.json')

In [8]:
answers = rag.load_answers(
    config.RAG_ANSWERS_PATH,
    test_set,
)

## Evaluation

- **Document retrieval:** *document hit@k* measures whether the gold article appears among the retrieved passage groups, while *document match@1* measures whether the first retrieved passage group comes from the gold article.
- **Chunk retrieval:** *exact chunk match@1* is a strict metric requiring the primary chunk of the first retrieved passage group to be the gold chunk. *Context chunk hit* measures whether the gold chunk appears anywhere in the final context supplied to the generative model, including primary retrieved chunks, merged adjacent chunks, and chunks added through neighbour expansion.
  
- **Answer evaluation:** `Qwen/Qwen3-4B-Instruct-2507` acts as an LLM judge. It receives the question, the complete retrieved context shown to the RAG model, the gold answer, and the RAG answer. It assigns two scores from 0 to 5:
  - **Correctness:** how accurately and completely the RAG answer addresses the question.
  - **Faithfulness:** whether every factual claim in the RAG answer is supported by the retrieved context.
- **Evaluation failures:** questions with an unusable gold answer are excluded from the judge averages. Judge outputs that cannot be parsed are counted separately instead of being assigned a score of zero.

The structured results are saved to `lfm_rag_answers_evaluated.json`. A human-readable version containing the summary metrics, individual answers, judge scores, and retrieved passages is saved to `evaluation_report.md`.

### Judge Prompt-Engineering

The judge prompt is designed to separate two properties that are often conflated in RAG evaluation: **answer correctness** and **grounding in the retrieved evidence**. A generated answer may be correct because of the model's internal knowledge but unsupported by the retrieved documents, or it may faithfully reproduce an irrelevant passage without properly answering the question. Scoring these dimensions independently makes this distinction explicit.


#### Allowing valid paraphrases

The correctness instructions explicitly state that an answer may receive a high score even if it differs in wording from the gold answer. This avoids the limitations of exact-match evaluation, which would incorrectly penalize semantically equivalent biomedical descriptions, synonymous terminology, and differences in phrasing.

At the same time, the prompt instructs the judge not to reward claims merely because they sound plausible. A claim must either match the gold answer or be supported by the retrieved context. This constraint is intended to reduce the tendency of an LLM judge to assign high scores to fluent but unverified responses.

#### Using an explicit faithfulness rubric

The faithfulness criterion includes a description for every score from 0 to 5. An explicit rubric improves consistency by reducing the amount of interpretation left to the judge:

- Scores of **4–5** represent fully or almost fully supported answers.
- A score of **3** permits one unsupported minor claim.
- Scores of **1–2** indicate increasingly substantial unsupported content.
- A score of **0** is reserved for contradictions or entirely unsupported answers.

The rubric focuses on factual claims rather than writing quality. This prevents style, length, or fluency from influencing the grounding score.

#### Requesting only structured scores

The output is restricted to:

```xml
<correctness>NUMBER</correctness>
<faithfulness>NUMBER</faithfulness>

In [9]:
print(prompts.JUDGE_TEMPLATE)

with LLM.load(config.QA_MODEL_ID) as judge:
    evaluated, summary = evaluation.evaluate(
        answers,
        judge,
    )

save_json(
    {"summary": summary, "records": evaluated},
    config.EVALUATION_PATH,
)
evaluation.write_markdown_report(
    evaluated,
    summary,
    config.REPORT_PATH,
)
evaluation.print_summary(summary)

BACKGROUND
You are an objective, precise AI evaluator grading a RAG system's answer to a biomedical question.

YOUR TASK
You are given the QUESTION, the RETRIEVED CONTEXT the system used, the GOLD ANSWER, and the RAG SYSTEM ANSWER.
Produce TWO scores, each from 0 to 5:

1) CORRECTNESS — how well the RAG SYSTEM ANSWER answers the QUESTION, judged against the GOLD ANSWER *and* the RETRIEVED CONTEXT. An answer may score 4-5 even if worded differently from the gold answer, but only if it is actually supported by the RETRIEVED CONTEXT or matches the gold answer. Do NOT reward plausible-sounding claims that are not in the context or gold answer.

2) FAITHFULNESS — whether every factual claim in the RAG SYSTEM ANSWER is supported by the RETRIEVED CONTEXT.
- 5: every claim is directly supported by the context
- 4: supported with trivially minor additions
- 3: mostly supported, one unsupported minor claim
- 2: several unsupported claims
- 1: mostly unsupported
- 0: contradicts the context or en

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!
=== Starting Grounded Qwen Evaluation ===
Row 1 | Correct: 5/5 | Faithful: 5/5 | Q: How can machine learning algorithms be u... | Gold: Machine learning algorithms ca...
Row 2 | Correct: 5/5 | Faithful: 5/5 | Q: What techniques are used to determine th... | Gold: SAXS and PDF are used to deter...
Row 3 | Correct: 5/5 | Faithful: 5/5 | Q: What is the importance of capturing deta... | Gold: Capturing detailed and diverse...
Row 4 | Correct: 5/5 | Faithful: 5/5 | Q: What machine learning algorithms are com... | Gold: Extreme gradient boosting (XGB...
Row 5 | Correct: 3/5 | Faithful: 5/5 | Q: What is a commonly used tool for managin... | Gold: Covidence...
Row 6 | Correct: 4/5 | Faithful: 3/5 | Q: What role does rheological analysis play... | Gold: Rheological analysis plays a c...
Row 7 | Correct: 4/5 | Faithful: 5/5 | Q: What is the difference between logistic ... | Gold: Logistic model trees use a sta...
Row 8 | Correct: 5/5 | Faithful: 4/5 | Q: What metrics a

In [ ]:
evaluated, summary = evaluation.load_evaluation(
    config.EVALUATION_PATH,
    answers,
)
evaluation.print_summary(summary)

In [2]:
retriever.close()

In [1]:
free_memory(report=True)

### Export artifacts

Set `EXPORT_ARCHIVES = True` to zip `document_repository/` and `vector_database/`, so they can be downloaded and reused in a later session.

In [ ]:
print("Wrote", make_zip(config.CHUNK_DIR))

In [9]:
print("Wrote", make_zip(config.VECTOR_DB_DIR))

Wrote /content/vector_database.zip


### Evaluation Summary

The RAG pipeline was evaluated on **20 biomedical questions** using retrieval-based metrics and `Qwen/Qwen3-4B-Instruct-2507` as an LLM judge. All 20 questions were successfully scored, with no invalid reference answers or judge-output parsing failures.

#### Overall results

| Metric | Result |
|---|---:|
| Document hit@k | 85% (17/20) |
| Document match@1 | 75% (15/20) |
| Exact chunk match@1 | 50% (10/20) |
| Context chunk hit | 75% (15/20) |
| Retrieval refusals | 0 |
| Mean correctness | 4.70 / 5 |
| Mean faithfulness | 4.85 / 5 |
| Questions scored | 20 / 20 |
| Invalid gold answers skipped | 0 |
| Judge parse failures | 0 |

#### Retrieval performance

The retriever found the correct source article among the returned passages for **17 of the 20 questions**. For **15 questions**, the highest-ranked passage came from the correct article.

Exact chunk retrieval was more difficult: the gold chunk was ranked first for only **10 questions**. However, the gold chunk appeared somewhere in the complete context for **15 questions**. The difference between exact chunk match@1 and context chunk hit indicates that reranking, adjacent-chunk merging, and neighbour expansion often recovered the relevant evidence even when the original gold chunk was not the highest-ranked primary result.

Three questions did not retrieve any passage from the gold document, while five did not include the exact gold chunk anywhere in the final context. Nevertheless, relevant information may still have appeared in other chunks or articles discussing the same topic.

#### Answer quality

The generated answers achieved a mean correctness score of **4.70 out of 5**:

- 15 answers received a correctness score of 5.
- 4 answers received a correctness score of 4.
- 1 answer received a correctness score of 3.

The mean faithfulness score was **4.85 out of 5**:

- 18 answers received a faithfulness score of 5.
- 1 answer received a faithfulness score of 4.
- 1 answer received a faithfulness score of 3.

These results suggest that the generator generally produced accurate answers grounded in the retrieved passages. The high answer scores despite lower exact-chunk retrieval also show that an answer does not always require the precise gold chunk: alternative passages may contain equivalent or sufficient supporting evidence.

#### Refusal behaviour

There were **no retrieval refusals**. Every question had at least one retrieved passage above the configured dense-similarity threshold. This is expected because the evaluation set contains only answerable questions generated from documents in the corpus.

### Possible improvements
 
#### 1. No negative or unanswerable questions
 
The current test set only contains answerable questions for which a gold chunk exists in the corpus. This means the retrieval and generation pipeline is never explicitly tested on questions that *cannot* be answered from the available documents. As a result:
 
- The system is not trained or evaluated to recognize out-of-scope or factually unanswerable queries.
- There is no signal on how often the model hallucinates when no relevant passage is retrieved.
- Users have no guarantee that the system will abstain rather than invent an answer.
 
**Suggested improvement:** add negative/unanswerable questions to the evaluation set. These can be generated by:

With such questions, a refusal mechanism (e.g., the `min_dense` threshold or an explicit classifier) can be calibrated and evaluated on both answerable and unanswerable inputs.
 
---
 
#### 2. Threshold calibration
 
The retrieval pipeline uses a single cosine-similarity floor, `min_dense`, to decide whether any document is relevant enough to answer a question. The default value of `0.45` is a reasonable starting point, but it is not tuned on the target test distribution:
 
- A threshold that is too low passes weak retrievals, which increases hallucination risk.
- A threshold that is too high causes excessive refusals, hurting recall on legitimate questions.
- The trade-off between retrieval coverage (`chunk_match_at_1`, `context_chunk_hit`) and refusals changes with the embedding model, corpus, and question style.
 
**Suggested improvement:** run a systematic threshold sweep over a validation set. For each candidate `min_dense`, measure:
- `chunk_match_at_1`
- `context_chunk_hit`
- `retrieval_refusals`
- judge-based correctness and faithfulness
 
Then select the threshold that maximizes correctness while keeping refusals within an acceptable range. If unanswerable questions are added, the threshold can be optimized for both precision (refusing bad queries) and recall (answering good queries).

### Use of AI 

This project used AI to help with coding, debugging, markdown formatting and rephrasing.

#### Coding prompt example

"Write a python function that takes as input XLM raw code and returns text."

#### Debugging prompt example

"Explain this error: 'AttributeError: 'HybridRetriever' object has no attribute 'position' in retrieval.py:108'"

#### Markdown formatting prompt example

"Summarise in a markdown table lfm_rag_answers_evaluated.json."

#### Rephrasing prompt example

"Rewrite this sentence: The choice of using LiquidAI/LFM2-1.2B-RAG was made because it is a good model for RAG."